# Interactive Simulation Experiments

Interactive widgets for exploring synthetic BPMN processes and RFC plots.

> Note: If no buttons are showing, the notebook may need a restart:

![Restart kernel and run all](images/restart_kernel_run_all.png)

## Simulation parameters

| Category | Parameter | Description |
|----------|-----------|-------------|
| General | Num. cases ($N$) | Number of simulated process instances (default: 10,000) |
| Control flow | Num. parallel paths ($m$) | Number of concurrent branches |
| | Num. XOR gates ($n$) | Number of exclusive-choice (XOR) stages per parallel branch |
| | Choices per XOR gate ($o$) | Number of branches at each XOR gate |
| | Loop probability ($p$) | Probability of taking the *loop back* path after the parallel join |
| | XOR probability distribution ($q$) | *equal*, *majority*, or *power_law* |
| | Power-law exponent ($\alpha$) | When $q=$*power_law* |
| Drift | Overlapping activity sets ($s$) | *overlapping* or *non_overlapping* activity names for $a$ and $b$ |

> Note: For *process change* (drift) experiments, `N` is the total number of cases in the combined log; half is generated from process `a` and half from process `b`.


In [1]:
x_max_in_graphs = None # None defaults to automatically set axis max
y_max_in_graphs = None # None defaults to automatically set axis max


## Interactive experiments


### Simple process structure


In [2]:
import ipywidgets as widgets
from IPython.display import display
from utils.custom_process_structure import control_flow_to_settings, run_simple_experiment
from utils.custom_process_structure.notebook_widgets import (
    GLOBAL_RANDOM_SEED,
    collect_control_flow,
    control_flow_widget_list,
    display_in_output,
    iter_simple_displays,
    make_N_slider,
    make_control_flow_sliders,
    show_objects,
)

simple_N = make_N_slider()
simple_cf = make_control_flow_sliders()
simple_run = widgets.Button(description="Run", button_style="primary")
simple_out = widgets.Output()

def _run_simple(_):
    simple_run.disabled = True
    try:
        def _render():
            cf = collect_control_flow(simple_cf)
            settings = control_flow_to_settings(cf, N=simple_N.value, seed=GLOBAL_RANDOM_SEED)
            result = run_simple_experiment(
                settings,
                trace_count=15,
                x_max_in_graphs=x_max_in_graphs,
                y_max_in_graphs=y_max_in_graphs,
            )
            show_objects(iter_simple_displays(result))
            stats = result["fit_stats"]
            print(f"alpha={stats['power_alpha']:.4f}, variants={len(result['event_log']['case'].drop_duplicates())}")
        display_in_output(simple_out, _render)
    finally:
        simple_run.disabled = False

simple_run.on_click(_run_simple)
display(widgets.VBox([simple_N, *control_flow_widget_list(simple_cf), simple_run, simple_out]))


### Process composition


In [3]:
import ipywidgets as widgets
from IPython.display import display
from utils.custom_process_structure import run_composition_experiment
from utils.custom_process_structure.notebook_widgets import (
    GLOBAL_RANDOM_SEED,
    collect_control_flow,
    control_flow_widget_list,
    display_in_output,
    iter_two_process_displays,
    make_N_slider,
    make_control_flow_sliders,
    show_objects,
)

comp_N = make_N_slider()
comp_a = make_control_flow_sliders({"alpha": 1.0})
comp_b = make_control_flow_sliders({"alpha": 3.0})
comp_accordion = widgets.Accordion(children=[
    widgets.VBox(control_flow_widget_list(comp_a)),
    widgets.VBox(control_flow_widget_list(comp_b)),
])
comp_accordion.set_title(0, "Process a")
comp_accordion.set_title(1, "Process b")
comp_run = widgets.Button(description="Run", button_style="primary")
comp_out = widgets.Output()

def _run_composition(_):
    comp_run.disabled = True
    try:
        def _render():
            settings = {"N": comp_N.value, "a": collect_control_flow(comp_a), "b": collect_control_flow(comp_b), "seed": GLOBAL_RANDOM_SEED}
            result = run_composition_experiment(
                settings,
                trace_count=15,
                x_max_in_graphs=x_max_in_graphs,
                y_max_in_graphs=y_max_in_graphs,
            )
            show_objects(iter_two_process_displays(result))
            print(f"cases={len(result['event_log']['case'].drop_duplicates())}")
        display_in_output(comp_out, _render)
    finally:
        comp_run.disabled = False

comp_run.on_click(_run_composition)
display(widgets.VBox([comp_N, comp_accordion, comp_run, comp_out]))


### Process change


In [4]:
import ipywidgets as widgets
from IPython.display import display
from utils.custom_process_structure import run_drift_experiment
from utils.custom_process_structure.notebook_widgets import (
    GLOBAL_RANDOM_SEED,
    collect_control_flow,
    control_flow_widget_list,
    display_in_output,
    iter_two_process_displays,
    make_N_slider,
    make_control_flow_sliders,
    make_overlap_selector,
    show_objects,
)

drift_N = make_N_slider(description="Total cases (N)")
drift_s = make_overlap_selector()
drift_a = make_control_flow_sliders({"n": 1, "o": 100, "q": "power_law", "alpha": 1.0, "p": 0.0, "m": 1})
drift_b = make_control_flow_sliders({"n": 1, "o": 100, "q": "power_law", "alpha": 3.0, "p": 0.0, "m": 1})
drift_accordion = widgets.Accordion(children=[
    widgets.VBox(control_flow_widget_list(drift_a)),
    widgets.VBox(control_flow_widget_list(drift_b)),
])
drift_accordion.set_title(0, "Process a")
drift_accordion.set_title(1, "Process b")
drift_run = widgets.Button(description="Run", button_style="primary")
drift_out = widgets.Output()

def _run_drift(_):
    drift_run.disabled = True
    try:
        def _render():
            settings = {"N": drift_N.value, "s": drift_s.value, "a": collect_control_flow(drift_a), "b": collect_control_flow(drift_b), "seed": GLOBAL_RANDOM_SEED}
            result = run_drift_experiment(
                settings,
                trace_count=15,
                x_max_in_graphs=x_max_in_graphs,
                y_max_in_graphs=y_max_in_graphs,
            )
            show_objects(iter_two_process_displays(result))
            N = drift_N.value
            N_a = N // 2
            N_b = N - N_a
            print(f"cases={len(result['event_log']['case'].drop_duplicates())} (a={N_a}, b={N_b})")
        display_in_output(drift_out, _render)
    finally:
        drift_run.disabled = False

drift_run.on_click(_run_drift)
display(widgets.VBox([drift_N, drift_s, drift_accordion, drift_run, drift_out]))
